In [7]:
"""
02 - Autoencoder on a 8x8 handwritten digits. 
"""
# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Scikit learn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [8]:
# Setting the PyTorch seed
SEED = 373
torch.manual_seed(SEED)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
# 1. Data
# Loading the dataset from load_digts | 1797 images
digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = digits.target                                           # y in autoencoder is not part of the loss

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
train_dl = DataLoader(TensorDataset(X_tr), batch_size=64, shuffle=True)
val_dl = DataLoader(TensorDataset(X_va), batch_size=512)

In [31]:
X

tensor([[0.0000, 0.0000, 0.3125,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.6250, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 1.0000, 0.5625, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0625,  ..., 0.3750, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1250,  ..., 0.7500, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6250,  ..., 0.7500, 0.0625, 0.0000]])

In [ ]:
# 2. Models

## 2.1 Autoencoder
class AutoEncoder(nn.Module):
    def __init__(self, d_in=64, d_hidden=128, d_latent=8):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_in), nn.Sigmoid(), # output in [0, 1] like the input
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def loss(self, x):
        x_hat, _ = self(x)
        return F.mse_loss(x_hat, x)


array([0, 1, 2, ..., 8, 9, 8], shape=(1797,))

In [45]:
a = [1, 2, 3]
a[-1] = 4
print(a)

[1, 2, 4]


In [ ]:
# Generic train loop

In [17]:
digits.data[0]

array([ 0.,  0.,  5., 13.,  9.,  1.,  0.,  0.,  0.,  0., 13., 15., 10.,
       15.,  5.,  0.,  0.,  3., 15.,  2.,  0., 11.,  8.,  0.,  0.,  4.,
       12.,  0.,  0.,  8.,  8.,  0.,  0.,  5.,  8.,  0.,  0.,  9.,  8.,
        0.,  0.,  4., 11.,  0.,  1., 12.,  7.,  0.,  0.,  2., 14.,  5.,
       10., 12.,  0.,  0.,  0.,  0.,  6., 13., 10.,  0.,  0.,  0.])